# N075 · 最近公共祖先与返回值语义

**目标：** 用子树是否包含目标定义递归返回信息。

**先修：** N071, N073。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 普通树LCA；BST路径；节点身份；目标存在性；路径法对照。

## 从问题到算法

最近公共祖先是同时位于两条根路径上且最深的节点。先记录每个节点的父节点，再从p向上建立祖先集合，从q向上首次命中的节点就是答案。这个版本明确处理目标缺失：返回None。BST版本可用键的分布直接定位分叉，但要求两目标确实属于树。

## 最小实现

**本章接口：** `lowest_common_ancestor(root, p, q)`、`lca_bst(root, p, q)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

def lowest_common_ancestor(root,p,q):
    if root is None: return None
    parent={root:None}; stack=[root]
    while stack and (p not in parent or q not in parent):
        node=stack.pop()
        for child in (node.left,node.right):
            if child is not None: parent[child]=node; stack.append(child)
    if p not in parent or q not in parent: return None
    ancestors=set(); node=p
    while node is not None: ancestors.add(node); node=parent[node]
    node=q
    while node not in ancestors: node=parent[node]
    return node

def lca_bst(root,p,q):
    low,high=sorted((p.val,q.val)); node=root
    while node:
        if node.val<low: node=node.right
        elif node.val>high: node=node.left
        else: return node
    return None

## 正确性、前提与复杂度

从q向上首次进入p的祖先集合，不可能有更低的公共祖先；所有公共祖先都在该节点以上。BST两目标位于同侧时祖先必在该侧，否则当前节点是最低分叉。

**代价：** 一般树O(n)时间空间；BST单次O(h)时间/O(1)辅助空间。比较身份而非值，一般树允许相同值节点。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([6,2,8,0,4,7,9,None,None,3,5])
p=r.left; q=r.left.right
show_table(['p','q','一般树LCA','BST LCA'],[(p.val,q.val,lowest_common_ancestor(r,p,q).val,lca_bst(r,p,q).val)])

p,q,一般树LCA,BST LCA
2,4,2,2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=tree_from_level([6,2,8,0,4,7,9,None,None,3,5])
assert lowest_common_ancestor(r,r.left,r.right) is r
assert lowest_common_ancestor(r,r.left,r.left.right) is r.left
assert lowest_common_ancestor(r,TreeNode(2),r.left) is None
nodes=[]; parent={r:None}; stack=[r]
while stack:
    n=stack.pop(); nodes.append(n)
    for c in (n.left,n.right):
        if c: parent[c]=n; stack.append(c)
def path(n):
    out=[]
    while n: out.append(n); n=parent[n]
    return out[::-1]
for p in nodes:
    for q in nodes:
        common=[a for a,b in zip(path(p),path(q)) if a is b]
        assert lowest_common_ancestor(r,p,q) is common[-1]
        assert lca_bst(r,p,q) is common[-1]
print('N075: 所有本章断言通过')

N075: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 扩展到目标节点可能缺失的自编任务。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释值相同不表示同一节点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 236. Lowest Common Ancestor of a Binary Tree（canonical）
- 235. Lowest Common Ancestor of a Binary Search Tree（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。